## Laboratorio 07 — Búsqueda Adversaria  

**Curso:** Inteligencia Artificial 2026  
**Fecha:** 04 de mayo de 2026

En este laboratorio se implementan algoritmos de búsqueda adversaria:  
- Minimax  
- Minimax con horizonte limitado  
- Poda α-β  
- Monte Carlo Tree Search (MCTS)  

El objetivo es analizar el impacto de la profundidad de búsqueda y el uso de métodos probabilísticos en juegos de suma cero.

In [3]:
import copy
import random
import math


### **1. Tic-Tac-Toe**

Implementar una clase `TicTacToeEngine` para realizar y visualizar un juego de Tic-Tac-Toe de 3×3 o 4×4.

**Objetivo:** Desarrollar un motor de juegos que compare la eficiencia de los algoritmos deterministas (Minimax/Alfa-Beta) frente a probabilísticos (MCTS) bajo distintas configuraciones de tablero.

Construir dos clases principales:

• Clase TicTacToeEngine: Esta clase representa el ”cerebro” y el estado del juego. No debe contener lógica de interfaz de usuario (inputs), solo procesamiento de datos. Métodos obligatorios a implementar:

– ``is_empty(row, col): ``Retorna si una celda está disponible.  
– ``get_moves(): Retorna ``lista de coordenadas (r, c) disponibles.  
– ``is_winner(player):`` Verifica si el jugador ha ganado.  
– ``evaluate(): ``Función heurística para tableros no terminados (asigna puntajes basados en proximidad a ganar).  
– ``minimax_pure(): ``Implementación exhaustiva (solo recomendada para 3 × 3).  
– ``minimax_limit(depth):`` Minimax que se detiene en un horizonte fijo y usa evaluate().  
– ``alpha_beta(depth, alpha, beta):`` Minimax optimizado con poda.  
– ``mcts(iterations, C):`` Monte Carlo Tree Search utilizando la fórmula UCT para selección. Fórmula UCT:

$$
\text{Score} = \text{mean\_win\_rate} + C \cdot \sqrt{\frac{\ln(\text{ParentVisits})}{\text{NodeVisits}}}
$$

In [ ]:
class TicTacToeEngine:
    """
    Motor de juego Tic-Tac-Toe de 3x3 o 4x4.
    No contiene lógica de interfaz de usuario (solo procesamiento de datos).
    X es el jugador maximizador, O es el minimizador.
    """

    def __init__(self, size=3):
        self.size = size
        self.board = [[' ' for _ in range(size)] for _ in range(size)]
        self.nodes_explored = 0  # Contador de nodos para medir eficiencia

    # ------------------------------------------------------------------ #
    #  Utilidades básicas del tablero                                       #
    # ------------------------------------------------------------------ #

    def is_empty(self, row, col):
        """Retorna True si la celda (row, col) está disponible."""
        return self.board[row][col] == ' '

    def get_moves(self):
        """Retorna lista de coordenadas (r, c) de celdas disponibles."""
        return [
            (r, c)
            for r in range(self.size)
            for c in range(self.size)
            if self.is_empty(r, c)
        ]

    def is_winner(self, player):
        """Verifica si el jugador indicado ha ganado (filas, columnas, diagonales)."""
        b = self.board
        n = self.size

        # Filas y columnas
        for i in range(n):
            if all(b[i][j] == player for j in range(n)):
                return True
            if all(b[j][i] == player for j in range(n)):
                return True

        # Diagonal principal
        if all(b[i][i] == player for i in range(n)):
            return True

        # Diagonal anti-principal
        if all(b[i][n - 1 - i] == player for i in range(n)):
            return True

        return False

    def is_terminal(self):
        """
        Verifica si el juego terminó.
        Retorna 'X', 'O', 'Tie' o None si el juego continúa.
        """
        if self.is_winner('X'):
            return 'X'
        if self.is_winner('O'):
            return 'O'
        if not self.get_moves():
            return 'Tie'
        return None

    # ------------------------------------------------------------------ #
    #  Función heurística                                                   #
    # ------------------------------------------------------------------ #

    def evaluate(self):
        """
        Heurística para tableros no terminados.
        Asigna puntajes basados en proximidad a ganar
        """
        state = self.is_terminal()
        if state == 'X':
            return 1000
        if state == 'O':
            return -1000
        if state == 'Tie':
            return 0

        b = self.board
        n = self.size
        score = 0

        # Recopilar todas las líneas del tablero
        lines = []
        for i in range(n):
            lines.append([b[i][j] for j in range(n)])   # fila i
            lines.append([b[j][i] for j in range(n)])   # columna i
        lines.append([b[i][i] for i in range(n)])        # diagonal principal
        lines.append([b[i][n - 1 - i] for i in range(n)])  # anti-diagonal

        for line in lines:
            x_count = line.count('X')
            o_count = line.count('O')

            # Línea pura de X (sin O) → potencial para X
            if x_count > 0 and o_count == 0:
                score += x_count ** 2

            # Línea pura de O (sin X) → potencial para O
            if o_count > 0 and x_count == 0:
                score -= o_count ** 2

        return score

    # ------------------------------------------------------------------ #
    #  Minimax exhaustivo (recomendado solo para 3×3)                      #
    # ------------------------------------------------------------------ #

    def minimax_pure(self, is_maximizing):
        self.nodes_explored += 1

        result = self.is_terminal()
        if result == 'X':
            return 1, None
        if result == 'O':
            return -1, None
        if result == 'Tie':
            return 0, None

        moves = self.get_moves()
        best_move = None

        if is_maximizing:
            best_val = -float('inf')
            for r, c in moves:
                self.board[r][c] = 'X'
                val, _ = self.minimax_pure(False)
                self.board[r][c] = ' '
                if val > best_val:
                    best_val, best_move = val, (r, c)
        else:
            best_val = float('inf')
            for r, c in moves:
                self.board[r][c] = 'O'
                val, _ = self.minimax_pure(True)
                self.board[r][c] = ' '
                if val < best_val:
                    best_val, best_move = val, (r, c)

        return best_val, best_move

    # ------------------------------------------------------------------ #
    #  Minimax con horizonte limitado                                       #
    # ------------------------------------------------------------------ #

    def minimax_limit(self, depth, is_maximizing):
        self.nodes_explored += 1

        result = self.is_terminal()
        if result is not None or depth == 0:
            return self.evaluate(), None

        moves = self.get_moves()
        best_move = None

        if is_maximizing:
            best_val = -float('inf')
            for r, c in moves:
                self.board[r][c] = 'X'
                val, _ = self.minimax_limit(depth - 1, False)
                self.board[r][c] = ' '
                if val > best_val:
                    best_val, best_move = val, (r, c)
        else:
            best_val = float('inf')
            for r, c in moves:
                self.board[r][c] = 'O'
                val, _ = self.minimax_limit(depth - 1, True)
                self.board[r][c] = ' '
                if val < best_val:
                    best_val, best_move = val, (r, c)

        return best_val, best_move

    # ------------------------------------------------------------------ #
    #  Alpha-Beta pruning                                                   #
    # ------------------------------------------------------------------ #

    def alpha_beta(self, depth, alpha, beta, is_maximizing):
        """
        Minimax optimizado con poda Alfa-Beta.
        """
        self.nodes_explored += 1

        result = self.is_terminal()
        if result is not None or depth == 0:
            return self.evaluate(), None

        moves = self.get_moves()
        best_move = None

        if is_maximizing:
            best_val = -float('inf')
            for r, c in moves:
                self.board[r][c] = 'X'
                val, _ = self.alpha_beta(depth - 1, alpha, beta, False)
                self.board[r][c] = ' '
                if val > best_val:
                    best_val, best_move = val, (r, c)
                alpha = max(alpha, best_val)
                if beta <= alpha:
                    break  
        else:
            best_val = float('inf')
            for r, c in moves:
                self.board[r][c] = 'O'
                val, _ = self.alpha_beta(depth - 1, alpha, beta, True)
                self.board[r][c] = ' '
                if val < best_val:
                    best_val, best_move = val, (r, c)
                beta = min(beta, best_val)
                if beta <= alpha:
                    break  # Poda alfa

        return best_val, best_move

    # ------------------------------------------------------------------ #
    #  Monte Carlo Tree Search con UCT                                      #
    # ------------------------------------------------------------------ #

    def mcts(self, iterations, C, player):
        moves = self.get_moves()
        if not moves:
            return 0, None

        # Estadísticas por movimiento raíz
        wins  = {m: 0.0 for m in moves}
        visits = {m: 0   for m in moves}
        total_visits = 0  # visitas del nodo padre (raíz)

        opponent = 'O' if player == 'X' else 'X'

        for _ in range(iterations):
            if total_visits == 0:
                selected = random.choice(moves)
            else:
                def uct_score(m):
                    n_i = visits[m]
                    if n_i == 0:
                        return float('inf')  # Nodo no visitado → explorar primero
                    mean_win_rate = wins[m] / n_i
                    exploration   = C * math.sqrt(math.log(total_visits) / n_i)
                    return mean_win_rate + exploration

                selected = max(moves, key=uct_score)

            # ── Simulación (play-out aleatorio desde el movimiento seleccionado) ──
            sim = TicTacToeEngine(self.size)
            sim.board = [row[:] for row in self.board] # Copia rápida de la matriz
            r, c = selected
            sim.board[r][c] = player          # Aplicar movimiento seleccionado

            curr = opponent
            while sim.is_terminal() is None:
                avail = sim.get_moves()
                mr, mc = random.choice(avail)
                sim.board[mr][mc] = curr
                curr = 'O' if curr == 'X' else 'X'

            # ── Retropropagación ──────────────────────────────────────
            result = sim.is_terminal()
            if result == player:
                wins[selected] += 1.0
            elif result == 'Tie':
                wins[selected] += 0.5

            visits[selected] += 1
            total_visits      += 1

        # Elegir el movimiento con más visitas (criterio robusto estándar en MCTS)
        best_move = max(moves, key=lambda m: visits[m])
        self.nodes_explored = total_visits
        return 0, best_move



• Clase GameLoop: Esta clase orquesta el flujo de la partida. Debe ser capaz de configurar una partida con los siguientes parámetros en su constructor.

– size: 3 ó 4.  
– mode: ”H-H” (Humano vs Humano), ”H-IA” (Humano vs IA), ”IA-IA” (IA contra IA).  
– starting_player: Quien realiza el primer movimiento (’H’ o ’IA’). Asumiremos que El primero el humano o la IA1 siempre juegan con ’X’, y el adversario con ’O’.  
– ia_configs: Un diccionario o estructura que defina para cada IA:  

  * Algoritmo a usar (minimax, alpha_beta, mcts).  
  * depth: Horizonte para algoritmos de límite (default 4 para el tablero de 4 × 4).  
  * N: Número de simulaciones para MCTS.  
  * C: Constante de exploración para UCT (default √2).  

### **2. Explosión Combinatoria:**

(a) En el Tic-Tac-Toe de 3 × 3, realizar lo siguiente:

• Una búsqueda minimax, desde el tablero vacío, variando el depth desde 1 a 9. Registrar el número de nodos visitados y el tiempo de ejecución.

• Repetir lo anterior, pero ahora implementando la poda α-β dentro del minimax.

(b) En el Tic-Tac-Toe de 4 × 4, realizar una búsqueda desde el tablero vacío con α-β variando el depth de 1 a 6. Registrar el número de nodos visitados, el tiempo de ejecución y el factor de ramificación efectivo:

$$
\sqrt[\text{depth}]{\text{nodos}}
$$

### **3. Duelo de Algoritmos (IA-IA):**
Enfrenten dos configuraciones en 20 partidas:

• IA-1: MCTS con N = 500 y C = √2.  
• IA-2: Minimax limitado a depth = 4 y poda α-β.  

Pregunta: ¿Cuál es más ”inteligente” en términos de ganar y cuál es más ”eficiente” en términos de tiempo por jugada?


### **4. Para pensar**
Imagine que la IA no debe tardar más de 1 segundo por jugada. Si el algoritmo es lento, ¿qué tipo de implementación o modificación a la estructura anterior debe hacerse para devolver la mejor jugada encontrada considerando la restricción de tiempo? Proponga sus ideas.